<a href="https://colab.research.google.com/github/hibahrehman25-lang/ML_inter_Task1/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hibahrehman25-lang/ML_inter_Task1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import duckdb
from google.colab import userdata

In [ ]:

HF_TOKEN = userdata.get("HF_TOKEN")

In [ ]:
con = duckdb.connect()

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
import duckdb
import pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
TYPE HUGGINGFACE,
TOKEN '{HF_TOKEN}'
);
""")

df = con.sql("""
SELECT
    report_date,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,

    CASE
        WHEN gsc_impressions > 0
        THEN (gsc_clicks * 100.0) / gsc_impressions
        ELSE NULL
    END AS ctr

FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'

WHERE
    gsc_data_available = TRUE
    AND gsc_impressions > 0
    AND gsc_avg_position > 0
""").df()

# Signal 1
position_bins = [0,3,10,20,float("inf")]
labels = ["1-3","4-10","11-20","21+"]

df["position_bucket"] = pd.cut(
    df["gsc_avg_position"],
    bins=position_bins,
    labels=labels,
    include_lowest=True
)

signal1 = (
    df.groupby("position_bucket", observed=False)
      .agg(mean_ctr=("ctr","mean"),
           n=("ctr","count"))
      .reset_index()
)

print(signal1)

# Signal 2
impression_bins = [0,100,1000,float("inf")]
impression_labels = ["0-100","101-1000","1000+"]

df["impression_bucket"] = pd.cut(
    df["gsc_impressions"],
    bins=impression_bins,
    labels=impression_labels,
    include_lowest=True
)

signal2 = (
    df.groupby("impression_bucket", observed=False)
      .agg(mean_ctr=("ctr","mean"),
           n=("ctr","count"))
      .reset_index()
)

print(signal2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  position_bucket  mean_ctr        n
0             1-3  0.491821   564173
1            4-10  0.347264  1456122
2           11-20  0.276991   519223
3             21+  0.128915   908354
  impression_bucket  mean_ctr        n
0             0-100  0.302258  2814461
1          101-1000  0.307079   601053
2             1000+  0.271542    32358


## Baseline Rule

A page is prioritized for CTR optimization if it has meaningful search visibility, receives sufficient impressions, and its CTR is lower than the average CTR of pages within the same search position bucket. This rule uses only current Search Console signals and is intended to identify pages where improving CTR could have the greatest impact.

### Reason Code

LOW_CTR_HIGH_VISIBILITY

### Action

CTR_OPTIMIZATION

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
expected_ctr = signal1.set_index("position_bucket")["mean_ctr"].to_dict()

high_impression_threshold = df["gsc_impressions"].quantile(0.75)

def calculate_score(row):
    score = 0

    if row["gsc_impressions"] >= high_impression_threshold:
        score += 40

    if row["position_bucket"] == "1-3":
        score += 30
    elif row["position_bucket"] == "4-10":
        score += 20

    expected = expected_ctr.get(row["position_bucket"],0)

    if row["ctr"] < expected:
        score += 30

    return score

df["score"] = df.apply(calculate_score,axis=1)

df["reason_code"] = "LOW_CTR_HIGH_VISIBILITY"
df["action"] = "CTR_OPTIMIZATION"

ranked_df = df.sort_values("score",ascending=False)

ranked_df = ranked_df[
[
"content_hash_id",
"score",
"reason_code",
"action",
"ctr",
"gsc_impressions",
"gsc_avg_position"
]
]

import os
os.makedirs("work/outputs",exist_ok=True)

ranked_df.to_csv(
"work/outputs/baseline_action_score.csv",
index=False
)

ranked_df.head(20)


,content_hash_id,score,reason_code,action,ctr,gsc_impressions,gsc_avg_position
32,content_347a278bb1a646f5,100,LOW_CTR_HIGH_VISIBILITY,CTR_OPTIMIZATION,0.000000,134,2.082090
3447834,content_854693c22689a059,100,LOW_CTR_HIGH_VISIBILITY,CTR_OPTIMIZATION,0.000000,128,1.523438
2150988,content_7de44dc794fc14ea,100,LOW_CTR_HIGH_VISIBILITY,CTR_OPTIMIZATION,0.000000,89,2.584270
414682,content_5b125fb8e13cf7a3,100,LOW_CTR_HIGH_VISIBILITY,CTR_OPTIMIZATION,0.314465,318,2.515723
414644,content_5320c6da44bed5ec,100,LOW_CTR_HIGH_VISIBILITY,CTR_OPTIMIZATION,0.000000,197,1.005076
414670,content_91e24512e75555e1,100,LOW_CTR_HIGH_VISIBILITY,CTR_OPTIMIZATION,0.000000,71,2.309859
414668,content_02621dd7ca617b9a,100,LOW_CTR_HIGH_VISIBILITY,CTR_OPTIMIZATION,0.000000,496,2.302419
414667,content_4af9bb9b9adade81,100,LOW_CTR_HIGH_VISIBILITY,CTR_OPTIMIZATION,0.000000,83,1.915663
414666,content_0bf9370493b84a79,100,LOW_CTR_HIGH_VISIBILITY,CTR_OPTIMIZATION,0.000000,771,1.577173
2151089,content_9ba844df8b7ee5e8,100,LOW_CTR_HIGH_VISIBILITY,CTR_OPTIMIZATION,0.000000,137,2.218978


## Ranked Queue

A transparent rule-based score was created using three signals:

- High impressions (high potential impact)
- Good search position (already visible in search)
- CTR below the average for its position bucket

Each page receives a score based on these signals and is ranked from highest to lowest priority. The ranked results are exported to:

work/outputs/baseline_action_score.csv

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:

ranked_df.head(20)

,content_hash_id,score,reason_code,action,ctr,gsc_impressions,gsc_avg_position
32,content_347a278bb1a646f5,100,LOW_CTR_HIGH_VISIBILITY,CTR_OPTIMIZATION,0.000000,134,2.082090
3447834,content_854693c22689a059,100,LOW_CTR_HIGH_VISIBILITY,CTR_OPTIMIZATION,0.000000,128,1.523438
2150988,content_7de44dc794fc14ea,100,LOW_CTR_HIGH_VISIBILITY,CTR_OPTIMIZATION,0.000000,89,2.584270
414682,content_5b125fb8e13cf7a3,100,LOW_CTR_HIGH_VISIBILITY,CTR_OPTIMIZATION,0.314465,318,2.515723
414644,content_5320c6da44bed5ec,100,LOW_CTR_HIGH_VISIBILITY,CTR_OPTIMIZATION,0.000000,197,1.005076
414670,content_91e24512e75555e1,100,LOW_CTR_HIGH_VISIBILITY,CTR_OPTIMIZATION,0.000000,71,2.309859
414668,content_02621dd7ca617b9a,100,LOW_CTR_HIGH_VISIBILITY,CTR_OPTIMIZATION,0.000000,496,2.302419
414667,content_4af9bb9b9adade81,100,LOW_CTR_HIGH_VISIBILITY,CTR_OPTIMIZATION,0.000000,83,1.915663
414666,content_0bf9370493b84a79,100,LOW_CTR_HIGH_VISIBILITY,CTR_OPTIMIZATION,0.000000,771,1.577173
2151089,content_9ba844df8b7ee5e8,100,LOW_CTR_HIGH_VISIBILITY,CTR_OPTIMIZATION,0.000000,137,2.218978


## Top-20 Review

The highest-ranked pages generally have strong search visibility, meaningful impressions, and CTR values below the average for their respective position buckets. These pages represent the strongest candidates for CTR optimization.

**Action:** CTR_OPTIMIZATION

**Reason Code:** LOW_CTR_HIGH_VISIBILITY

**Confidence:** Medium to High

### What would make these recommendations wrong?

- The page primarily ranks for branded queries where CTR naturally differs.
- Seasonal search demand temporarily affected user behavior.
- SERP features (featured snippets, AI Overviews, ads, etc.) reduced organic CTR.
- The page is newly published and performance has not yet stabilized.
- The page already has an optimized title and meta description, meaning low CTR may not be easily improved.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
print("Signal 1")
print(signal1)

print("\nSignal 2")
print(signal2)

print("\nTotal ranked pages:", len(ranked_df))


Signal 1
  position_bucket  mean_ctr        n
0             1-3  0.491821   564173
1            4-10  0.347264  1456122
2           11-20  0.276991   519223
3             21+  0.128915   908354

Signal 2
  impression_bucket  mean_ctr        n
0             0-100  0.302258  2814461
1          101-1000  0.307079   601053
2             1000+  0.271542    32358

Total ranked pages: 3447872


## Weak Picks

Some high-scoring pages may still be weak recommendations because CTR is influenced by factors beyond impressions and position. Search intent, branded traffic, seasonality, and SERP features may reduce CTR even when the page performs well.

## Leakage Check

No future information or label-derived features were used.

This baseline only uses current observed signals:

- gsc_impressions
- gsc_clicks
- calculated CTR
- gsc_avg_position

The following were intentionally excluded:

- trend_direction
- trend_pct
- is_declining_label
- future windows
- product-generated flags

This makes the baseline suitable for comparison with the Week 5 machine learning model.

## Self-check

Before you submit, confirm each line honestly:

- [✔️] Every section above is filled — markdown thinking AND the code that backs it
- [✔️] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔️] No client names, URLs, or private queries anywhere
- [✔️] My claims use careful words: observed, measured, directional, decision-support
- [✔️] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.